In [22]:
import pathlib
import pandas as pd

import mne
from mne.preprocessing.nirs import optical_density

%load_ext watermark
%watermark --iversions

panel     : 0.13.0
mne       : 1.0.2
mne_nirs  : 0.2.1
numpy     : 1.21.5
matplotlib: 3.5.1
re        : 2.2.1
networkx  : 2.7.1
xarray    : 0.20.1
datashader: 0.14.0
colorcet  : 3.0.0
hvplot    : 0.7.3
holoviews : 1.14.8
pandas    : 1.4.2



In [5]:
dataPath = pathlib.Path('../data/finger_tapping/2021-10-01_002/2021-10-01_002.snirf')
raw = mne.io.read_raw_snirf(dataPath, verbose=True, preload=True)

Loading C:\Users\49171\Documents\Src\fnirz\notebooks\..\data\finger_tapping\2021-10-01_002\2021-10-01_002.snirf
Reading 0 ... 2761  =      0.000 ...   271.406 secs...


## transform data into long DF

In [12]:
# normalize the raw data then get it into a tidy Pandas DataFrame 
raw_od = optical_density(raw) # essentially log(data/mean(data))
data, times = raw_od.get_data(return_times=True) # data is (n_channels, n_times)
df = pd.DataFrame(data=data.T, index=times, columns=raw_od.ch_names).rename_axis('time')

In [13]:
df.head(2)

,S1_D1 760,S1_D1 850,S1_D3 760,S1_D3 850,S2_D1 760,S2_D1 850,S2_D2 760,S2_D2 850,S2_D4 760,S2_D4 850,...,S7_D4 760,S7_D4 850,S7_D6 760,S7_D6 850,S7_D7 760,S7_D7 850,S8_D5 760,S8_D5 850,S8_D7 760,S8_D7 850
time,,,,,,,,,,,,,,,,,,,,,
0.0000,-0.037738,-0.019991,0.039085,0.038791,-0.063061,-0.047853,0.033341,0.038849,0.026850,0.031370,...,-0.000455,0.001958,-0.004253,0.004660,-0.016381,-0.014141,-0.026197,-0.012854,-0.007756,0.005374
0.0983,-0.034953,-0.015961,0.040429,0.040715,-0.061259,-0.044363,0.037958,0.041879,0.028124,0.032178,...,-0.001007,0.002495,-0.004761,0.005312,-0.016593,-0.013357,-0.024433,-0.011169,-0.007260,0.006069


In [14]:
# seperate the channel and wavelength column levels
ch_wl = [(ch.split(' ')[0], int(ch.split('_')[1].split(' ')[1])) for ch in df.columns.values]
cindex = pd.MultiIndex.from_tuples(ch_wl, names=["channel", "wavelength"])
df.columns = cindex
df.head(2)

channel        S1_D1               S1_D3               S2_D1            \
wavelength       760       850       760       850       760       850   
time                                                                     
0.0000     -0.037738 -0.019991  0.039085  0.038791 -0.063061 -0.047853   
0.0983     -0.034953 -0.015961  0.040429  0.040715 -0.061259 -0.044363   

channel        S2_D2               S2_D4            ...     S7_D4            \
wavelength       760       850       760       850  ...       760       850   
time                                                ...                       
0.0000      0.033341  0.038849  0.026850  0.031370  ... -0.000455  0.001958   
0.0983      0.037958  0.041879  0.028124  0.032178  ... -0.001007  0.002495   

channel        S7_D6               S7_D7               S8_D5            \
wavelength       760       850       760       850       760       850   
time                                                                     
0.0000     -0.004253  0.004660 -0.016381 -0.014141 -0.026197 -0.012854   
0.0983     -0.004761  0.005312 -0.016593 -0.013357 -0.024433 -0.011169   

channel        S8_D7            
wavelength       760       850  
time                            
0.0000     -0.007756  0.005374  
0.0983     -0.007260  0.006069  

[2 rows x 44 columns]

In [16]:
# create offset array to nicely stack timeseries traces in same plot
offsetL = []
chYticks = []
for i, d in enumerate(df.groupby(axis=1, level='channel')):
    i = i*.1
    firstVal = d[1].values[0,:]
    offsetL.append(i - firstVal)
    chYticks.append((i, d[0]))
offsetAr = np.concatenate(offsetL, axis=0)
dfc = df.add(offsetAr)

# stack column levels into a long df to play nicely with holoviz
dfc = pd.DataFrame(dfc.stack(level=[0,1]), columns=['amplitude'])
dfc = dfc.reorder_levels(["channel", "wavelength", "time"])
dfc.head(2)

amplitude
channel wavelength time           
S1_D1   760        0.0         0.0
        850        0.0         0.0

## Viz

In [17]:
# Make curves plot
curvesDict = {i: hv.Curve(c, 'time', 'amplitude').opts(color='black', line_width=1) for i, c in dfc.groupby(['channel', 'wavelength'])}
linespread = datashade(hv.HoloMap(hv.NdOverlay(curvesDict, ['channel', 'wavelength']), ['channel', 'wavelength']).overlay('channel'), aggregator=ds.any(), cmap='black').opts(tools=['hover'])
linespread = linespread.opts(hv.opts.RGB(width=800, height=600, padding=0, yticks=chYticks, fontsize={'yticks':6}, ylabel='channel'))
# linespread = hv.HoloMap(hv.NdOverlay(curvesDict, ['channel', 'wavelength']), ['channel', 'wavelength']).overlay('channel').opts(width=800, height=600, padding=0, yticks=chYticks, fontsize={'yticks':6}, ylabel='channel', show_legend=False, tools=['hover'])

In [18]:
# Make triggers plot
triggersOnset = raw.annotations.onset
triggersDuration = raw.annotations.duration
triggers = pd.DataFrame.from_dict({'id':raw.annotations.description.astype('int'), 'start':triggersOnset, 'duration':triggersDuration})
condition = {1:'Right', 2:'Left'}
triggers['type'] = triggers.id.map(condition)
triggers['duration'] = 10
triggers['end'] = triggers['start'] + triggers['duration']
trigDict = {}
colors = ['purple', 'green']
for i,t in triggers.iterrows():
    trigDict[i] = hv.VSpan(t.start,t.end).opts(color=colors[t.id-1], alpha=.15)
trigHM = hv.HoloMap(trigDict).overlay()

In [19]:
# make channel plot
chs = raw.info['chs']
chPos = []
for ci, ch in enumerate(chs):
    chPos.append({'channel': ch['ch_name'].split(' ')[0]})
    chPos[-1]['sourceX'] = ch['loc'][3]
    chPos[-1]['sourceY'] = ch['loc'][4]
    chPos[-1]['detectorX'] = ch['loc'][6]
    chPos[-1]['detectorY'] = ch['loc'][7]

chDF = pd.DataFrame.from_dict(chPos).set_index('channel')
chPosTup = [[c, (i.sourceX, i.sourceY), (i.detectorX, i.detectorY)] for c,i in chDF.iterrows()]
pathD = {c: hv.Path([s,d]).opts(tools=(['hover'])) for c,s,d in chPosTup}
chPathPlot = hv.HoloMap(pathD, 'channel').overlay()

In [20]:
# make optode plot
chs = raw.info['chs']
optodeDict = {}

for ci, ch in enumerate(chs):
    sourceStr = ch['ch_name'].split('_')[0]
    sourceInt = [int(s) for s in re.findall(r'\d+', sourceStr)][0]
    sourcePos = ch['loc'][3:6]
    sourceX = ch['loc'][3]
    sourceY = ch['loc'][4]
    optodeDict[sourceStr] = {'id': sourceStr, 'type': 'source', 'typeNum': sourceInt, 'pos': sourcePos, 'x': sourceX, 'y':sourceY}
    
    detectorStr = ch['ch_name'].split('_')[1].split(' ')[0]
    detectorInt = [int(s) for s in re.findall(r'\d+', detectorStr)][0]
    detectorPos = ch['loc'][6:9]
    detectorX = ch['loc'][6]
    detectorY = ch['loc'][7]
    optodeDict[detectorStr] = {'id': detectorStr, 'type': 'detector', 'typeNum': detectorInt, 'pos': detectorPos, 'x':detectorX, 'y':detectorY}

optodeDF = pd.DataFrame.from_dict(optodeDict).T  
optodeLayout = hv.Points(optodeDF, kdims=['x','y'], vdims=['id', 'type']).opts(color='type', size=10, width=500, cmap=['#ff0000', '#0000ff'], tools=['hover'], show_legend=False)

In [21]:
trigHM * linespread + optodeLayout * chPathPlot

:Layout
   .DynamicMap.I :DynamicMap   [wavelength]
      :Overlay
         .NdOverlay.I :NdOverlay   [Default]
            :VSpan   [x,y]
         .RGB.I       :RGB   [time,amplitude]   (R,G,B,A)
   .Overlay.I    :Overlay
      .Points.I    :Points   [x,y]   (id,type)
      .NdOverlay.I :NdOverlay   [channel]
         :Path   [x,y]